J3 — Comparer sans se mentir, puis ce qu’on fait des défauts trouvés

**Séance 3.** La chaîne, vous la connaissez : lire, explorer, entraîner, soumettre. Aujourd’hui,
elle sert deux fois. D’abord à **comparer** : un score, contre quoi ? Puis à **décider** de ce que
l’exploration a trouvé — les trous, les colonnes inutiles, les valeurs impossibles.

> **La règle :** une décision — de modèle ou de nettoyage — se **mesure**. Un effet nul est un
> résultat : il dit que cette décision-là ne se joue pas là.

**Le fil :** on ne nettoie pas ce qu’on ne sait pas mesurer. La barre de bruit est posée dans la
première moitié. C’est elle qui rend lisible le « le score ne bouge pas » de la seconde.

**Ce que le devoir 2 vous demandera :** les baselines et le tableau à quatre colonnes. Le journal
de nettoyage (décision · choix · effet mesuré). Et le **pipeline**, qui va de la table brute à la
prédiction.

**La table de travail n’est pas reconstruite ici** : la mise en place ci-dessous est **extraite de
`notebook_J1.ipynb`**, qui reste la source unique. Il n’existe qu’une seule définition de
`matrice()` dans tout le cours.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb`**, qui reste la source unique de la table
de travail, de la référence et du modèle. Exécutez-les de haut en bas.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5).
Le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [ ]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [ ]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"

print("le dossier des données trouvé :", DOSSIER)


In [ ]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"

print("les deux tableaux \n")
print("train :", train.shape, "(lignes, colonnes)")
print("test  :", test.shape, "(lignes, colonnes)")

print("les cinq premières lignes \n")
display(train.head())


### La table de travail


In [ ]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)

print("la table de travail \n")
print("colonnes conservées :", X.shape[1])
print("colonnes écartées (cardinalité trop forte) :", TROP_FIN)
print("colonnes catégorielles :", CAT)


### La cible, et la référence


In [ ]:
print("la distribution de la cible \n")
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])
display(repartition)

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())
print(f"la référence (répondre toujours la classe majoritaire) : {reference:.2f} %")


In [ ]:
# `y` entre ici ; la référence a déjà été calculée avec la distribution ci-dessus —
# une seule définition, donc un seul chiffre imprimé (elle était calculée deux fois).
y = train[CIBLE]


### Le rituel : annoncer avant de mesurer


In [ ]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))


### Ce qu’un score doit battre

**82,51 %** ne dit pas qu’un modèle est bon. Cela dit qu’il fait un peu mieux que **81,75 %**,
la réponse constante. Un score ne prend son sens **que** contre un concurrent nommé. En production,
ce concurrent est rarement un autre modèle : c’est la **règle métier** en place. Elle tient en
trois lignes, personne ne la maintient, et personne n’ose la remplacer.

Ce qu’on regarde : **quatre** baselines mesurées sur le **même** jeu de validation. La réponse
constante. Une règle métier bâtie sur le délai. Une seconde règle lue dans l’historique du patient.
La famille linéaire. Puis l’écart du modèle **à la réponse constante**, exprimé en barres de bruit.
Un écart plus petit que la barre n’est pas un gain : c’est du tirage.

Les valeurs essayées s’arrêtent avant le délai le plus long de la table. Au-delà, la règle
n’annoncerait plus rien — et ne rien annoncer donne la meilleure exactitude sur cette cible rare.

Une chose que la mesure montre : la règle métier **perd** contre la réponse constante. Et la règle
qui ne dirait jamais rien la **vaudrait exactement**.

Ce que cette section **ne tranche pas** : ce que cela dit d’une exactitude sur une cible rare. La
métrique de décision n’est pas encore posée, et c’est elle qui trancherait.


In [ ]:
# les baselines, sur le même jeu de validation — et la barre qui décide
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

# 1) la réponse constante : le concurrent le plus bête, et le plus dur à battre
constante = 100 * max(yval.mean(), 1 - yval.mean())

# 2) la règle métier : « j'annonce une absence si le délai dépasse N jours ».
#    N se choisit sur les lignes d'apprentissage, pas sur celles de contrôle.
delai_max = int(Xtr['delai_jours'].max())
taux_par_seuil = {s: float(((Xtr['delai_jours'] > s).astype(int) == ytr).mean())
                  for s in (0, 1, 3, 7, 14, 30) if s < delai_max}
seuil_appris = max(taux_par_seuil, key=taux_par_seuil.get)
regle = (Xval['delai_jours'] > seuil_appris).astype(int)

# et la règle qui ne dirait jamais rien ? c'est la réponse constante, au point près
silence = (Xval['delai_jours'] > delai_max).astype(int)

# 3) la famille linéaire : elle a besoin de nombres, on lui en prépare — la recette des
#    séances précédentes (newton-cholesky : le solveur qui converge sur cette table)
X_n = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_n.loc[Xtr.index], X_n.loc[Xval.index]
lineaire = LogisticRegression(solver='newton-cholesky').fit(Xtr_n, ytr)

# 4) le modèle du cours, celui des séances précédentes
modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)

# la barre : l'erreur-type de l'exactitude sur CE jeu de validation
barre = 100 * ((constante / 100) * (1 - constante / 100) / len(yval)) ** 0.5

baselines = pd.DataFrame([
    {'baseline': 'toujours présent',
     'exactitude (%)': round(constante, 2)},
    {'baseline': f'règle métier — absent si le délai dépasse {seuil_appris} j',
     'exactitude (%)': round(100 * (regle == yval).mean(), 2)},
    {'baseline': 'absent si ce patient a déjà manqué un rendez-vous',
     'exactitude (%)': round(100 * ((Xval['taux_absence_passe'] > 0).astype(int) == yval).mean(), 2)},
    {'baseline': 'la famille linéaire',
     'exactitude (%)': round(100 * lineaire.score(Xval_n, yval), 2)},
    {'baseline': 'le modèle du cours',
     'exactitude (%)': round(100 * modele.score(Xval, yval), 2)},
])
display(baselines)

ecart = 100 * modele.score(Xval, yval) - constante
print(f'→ le modèle dépasse la réponse constante de {ecart:+.2f} point,')
print(f'  soit {ecart / barre:.1f} erreurs-types de ce jeu de validation ({barre:.2f} point).')
print(f'  la règle métier, elle, fait {100 * (regle == yval).mean():.2f} % : elle alerte sur')
print(f'  {int(regle.sum())} lignes et perd contre la réponse constante.')
print(f'  et la règle qui ne dirait jamais rien ferait {100 * (silence == yval).mean():.2f} % —')
print('  exactement la réponse constante. Une règle en place se mesure avant d’être opposée')
print('  à un modèle ; ce que cela dit d’une exactitude reste ouvert.')


> **Question →** `Devoir_maison_2.ipynb`, **question 1 (carnet J3)** — les baselines de votre main, l'écart divisé par la barre.


### Le test est sacré, et le piège du public

Un leaderboard public se lit comme une mesure. C’est un **échantillon**, et un petit. Sur mille
lignes, deux modèles qui valent la même chose n’obtiennent pas le même score. Choisir le premier
du public, c’est souvent choisir **le plus chanceux**. Le piège n’est pas de soumettre : c’est de
**décider** sur ce que le public montre.

Ce qu’on regarde : douze variantes jugées sur un public de **1 000 lignes**, puis re-mesurées sur
**3 000 lignes que personne n’a regardées**. Les deux classements sont-ils le même ? La cellule
prend **≈ 2 minutes** : c’est le prix de la démonstration, et il se paie une fois.

Ce que cette section **ne tranche pas** : laquelle de ces douze variantes mériterait d’être
soumise. Choisir, c’est décider sur un échantillon — et c’est ce que la suite mesure.


In [ ]:
# le leaderboard, monté en petit : 1 000 lignes de public, 3 000 de privé jamais regardées
import numpy as np

# le rituel du cours : on écrit ce qu'on attend avant de mesurer — sinon `noter` refuse la ligne
prevoir("le classement du public",
        "le premier du public reste le premier du privé",
        "le public est un petit échantillon : son classement est du bruit")

Xdev, Xreste, ydev, yreste = train_test_split(X, y, test_size=4000, stratify=y, random_state=0)
Xpub, Xpriv, ypub, ypriv = train_test_split(Xreste, yreste, test_size=3000, stratify=yreste, random_state=0)

reglages = ((20, 3), (20, 5), (20, 10), (20, 31), (40, 5), (40, 10),
            (60, 5), (60, 31), (80, 10), (100, 5), (150, 10), (200, 31))
variantes = []
for max_iter, feuilles in reglages:
    m = HistGradientBoostingClassifier(max_iter=max_iter, max_leaf_nodes=feuilles,
                                       random_state=0).fit(Xdev, ydev)
    variantes.append({'variante': f'max_iter={max_iter} · feuilles={feuilles}',
                      'public (%)': round(100 * m.score(Xpub, ypub), 2),
                      'privé (%)': round(100 * m.score(Xpriv, ypriv), 2)})

tableau = pd.DataFrame(variantes).sort_values('public (%)', ascending=False).reset_index(drop=True)
display(tableau)

premier, dernier = tableau.iloc[0], tableau.iloc[-1]
print(f'→ le premier du public : {premier["public (%)"]:.2f} % au public, {premier["privé (%)"]:.2f} % au privé.')
print(f'  le dernier du public  : {dernier["public (%)"]:.2f} % au public, {dernier["privé (%)"]:.2f} % au privé.')
correlation = float(np.corrcoef(tableau['public (%)'], tableau['privé (%)'])[0, 1])
print(f'  corrélation du public et du privé, sur ces douze variantes : {correlation:.3f}')
quatre_public = tableau.head(4)['variante'].tolist()
quatre_prive = pd.DataFrame(variantes).sort_values('privé (%)', ascending=False).head(4)['variante'].tolist()
print(f'  parmi les quatre premiers du public, {len([v for v in quatre_prive if v in quatre_public])} sont dans les quatre premiers du privé.')
erreur_type_public = 100 * (ypub.mean() * (1 - ypub.mean()) / len(ypub)) ** 0.5
etendue_public = tableau['public (%)'].max() - tableau['public (%)'].min()
print(f'  l erreur-type d une exactitude sur {len(ypub)} lignes : {erreur_type_public:.2f} point —')
print(f'  pour des variantes étalées sur {etendue_public:.2f} point. Le classement tient dans')
print('  le bruit de l échantillon qui le produit.')
print('→ un classement qui ne se retrouve pas d’un jeu à l’autre ne classe rien :')
print('  il trie le hasard, et l’on croit avoir choisi le meilleur modèle.')

# et la ligne d'essai : l'attendu était écrit avant, le mesuré arrive ici
noter("le classement du public", "douze variantes, un public de 1 000 lignes",
      "le premier du public reste premier", float(tableau.loc[0, "privé (%)"]))


> **Question →** `Devoir_maison_2.ipynb`, **question 2 (carnet J3)** — le classement du public tient-il ?


### Un gain de 0,2 point est-il un gain ?

Le même modèle, entraîné deux fois, ne rend pas le même score : le découpage change, donc la
mesure change. La question n’est jamais « le score a-t-il monté ? », mais « **de combien** a-t-il
bougé, et ce changement fait-il plus que ce que le hasard fait tout seul ? ».

La barre du cours est l’**erreur-type d’une exactitude** : **± 0,22 point** sur 29 979 lignes de
validation. Un écart ne se lit qu’à partir du **double**, soit 0,44 — deux erreurs-types,
c’est-à-dire environ 95 % des tirages sous une loi normale. Ce n’est pas une « règle du cours »
sortie de nulle part : c’est la dispersion d’une mesure sur cet échantillon-là, et elle se calcule.

Ce qu’on regarde : cinq graines sur le même modèle, c’est-à-dire la dispersion que personne ne
choisit. Puis deux modèles comparés sur **un seul et même** découpage.

La dispersion des cinq tirages est **plus petite** que la barre, et c’est normal. Les cinq
découpages partagent les mêmes lignes, donc leurs jeux de contrôle se recouvrent. Ce ne sont pas
les mêmes grandeurs : la barre juge un écart à la réponse constante, pas un écart entre deux
modèles.

Ce que cette section **ne tranche pas** : deux modèles jugés sur le même échantillon se testent
par un test **apparié**. La notion est nommée ici, elle n’est pas développée.


In [ ]:
# cinq découpages, le même modèle : la dispersion que personne ne choisit
graines = []
for graine in range(5):
    X_appris, X_controle, y_appris, y_controle = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=graine)
    m = HistGradientBoostingClassifier(random_state=0).fit(X_appris, y_appris)
    graines.append(round(100 * m.score(X_controle, y_controle), 2))

ecart_type = float(np.std(graines, ddof=1))
print('les cinq scores :', graines)
print(f'  moyenne {np.mean(graines):.2f} · écart-type {ecart_type:.3f} · étendue {max(graines) - min(graines):.2f} point')
print(f'  à comparer à la barre d’une exactitude sur {len(yval)} lignes : {barre:.2f} point.')
print('  plus petite, et c’est normal : les cinq tirages partagent les mêmes lignes, donc leurs')
print('  jeux de contrôle se recouvrent — deux grandeurs différentes, qui ne jugent pas la même')
print('  chose.')

# et deux modèles, sur le même découpage — la seule comparaison honnête
# `early_stopping=False` : sans lui, l'arrêt anticipé borne les deux au même arbre.
court = HistGradientBoostingClassifier(max_iter=60, random_state=0, early_stopping=False).fit(Xtr, ytr)
long = HistGradientBoostingClassifier(max_iter=120, random_state=0, early_stopping=False).fit(Xtr, ytr)
ecart = 100 * long.score(Xval, yval) - 100 * court.score(Xval, yval)
print(f'  deux modèles ({court.n_iter_} arbres contre {long.n_iter_}), un seul découpage : {ecart:+.2f} point')
print(f'  la barre vaut {barre:.2f} point, son double {2 * round(barre, 2):.2f} — la règle du cours.')
print('→ un écart plus petit que la barre n’apprend rien : il faut le même découpage, et la barre.')


> **Question →** `Devoir_maison_2.ipynb`, **question 3 (carnet J3)** — deux modèles, un seul découpage.


### Le tableau à quatre colonnes

C'est le tableau qui ferme la comparaison : **modèle · validation · public · écart**. Il ne sert
pas à désigner un vainqueur, il sert à lire l'écart entre ce que vous mesurez chez vous et ce que
le public affiche. Quand la validation monte et que le public stagne, ce n'est pas le modèle qui a
progressé : c'est vous qui avez appris le public.

Ce qu'on regarde : le tableau rempli avec ce qui est mesurable ici — la colonne du public
n'existe que pour un modèle **effectivement soumis**.

Ce que cette section **ne tranche pas** : quelle ligne de ce tableau mérite d'être déployée. Le
tableau lit un écart, il ne désigne pas un vainqueur.


In [ ]:
# le tableau à quatre colonnes : les scores mesurés ici, et le public relevé à la soumission
lignes = []
for nom, decision in (('toujours présent', np.zeros(len(yval), dtype=int)),
                      (f'règle métier (délai > {seuil_appris} j)', regle),
                      ('le modèle du cours', modele.predict(Xval))):
    lignes.append({'modèle': nom,
                   'validation (%)': round(100 * (decision == yval).mean(), 2)})
tableau_quatre = pd.DataFrame(lignes)
# le public ne se remplit que pour un modèle réellement soumis : ici, un seul
tableau_quatre['public (%)'] = [None, None, 81.81]   # le seul score public du cours : celui de la séance 1
tableau_quatre['écart'] = (tableau_quatre['public (%)'] - tableau_quatre['validation (%)']).round(2)
display(tableau_quatre)
print('→ une colonne vide n’est pas un oubli : c’est un modèle qui n’a pas été soumis.')
print('  Un écart sain est petit. Un écart qui grandit veut dire que le public a servi de guide.')


> **Question →** `Devoir_maison_2.ipynb`, **question 4 (carnet J3)** — votre tableau à quatre colonnes.


### Le plafond — ce que les données permettent

La séance précédente a laissé une question ouverte. On a entraîné un modèle, il bat la référence.
Jusqu’où peut-on monter ?

**Trois familles très différentes** — des arbres entraînés par corrections successives, un arbre
unique, un modèle linéaire — donnent ici presque le même chiffre. Ce n’est pas un hasard. Quand
trois familles qui ne partagent pas leur mécanisme plafonnent au même endroit, le plafond n’est
pas dans le modèle. **Il est dans les données** — et le savoir évite de chercher pendant six mois
ce qui n’existe pas.

Ce que cette section **ne tranche pas** : si une autre famille ferait mieux. Elle mesure un
plafond, elle n’épuise pas le catalogue.


In [ ]:
# les trois familles : elles entrent ici
from sklearn.tree import DecisionTreeClassifier

# le découpage, la table d'indicatrices et le linéaire viennent de la section précédente :
# ce carnet n'en donne qu'une définition, et c'est elle qui sert partout.
print("trois familles, la même table, la même mesure \n")
scores = {}
for nom, famille, donnees in (
        ("arbres (boosting)", HistGradientBoostingClassifier(random_state=0), (Xtr, Xval)),
        ("un arbre unique", DecisionTreeClassifier(random_state=0, max_depth=8), (Xtr_n, Xval_n)),
        ("modèle linéaire", LogisticRegression(solver="newton-cholesky"), (Xtr_n, Xval_n))):
    a, b = donnees
    scores[nom] = 100 * famille.fit(a, ytr).score(b, yval)
    print(f"  {nom:<22} {scores[nom]:6.2f} %")

print("\nl'écart entre la meilleure et la moins bonne famille :",
      round(max(scores.values()) - min(scores.values()), 2), "point(s).")
print("→ le plafond n'est pas dans le modèle.")


> **Question →** `Devoir_maison_2.ipynb`, **question 5 (carnet J3) (1/2)** — le plafond, et les étiquettes.


#### Et l’autre cause du plafond : des étiquettes fausses

Le plafond peut venir des données. Il peut aussi venir des **étiquettes elles-mêmes**. Personne
n’annote parfaitement, et un annotateur qui se trompe ne se trompe pas comme un modèle.

On le simule : on **retourne une part des étiquettes d’entraînement** (0 %, 15 %, 30 %), et on
mesure sur une validation **intacte**. Sinon on mesurerait le bruit, pas ce qu’il coûte. La
question n’est pas « le score baisse-t-il ? » mais « **sur quel modèle** le bruit mord-il ? ». La
réponse se lit sur deux colonnes : les deux modèles de la séance précédente cohabitent ici.

Ce que cette section **ne tranche pas** : d’où viennent les étiquettes fausses. Un carnet ne
corrige pas l’annotation.


In [ ]:
# le bruit d'étiquette : la même table, avec des étiquettes d'entraînement abîmées.
# On mesure sur une validation intacte — sinon on mesure le bruit, pas ce qu'il coûte.
brouille = []
for part in (0.0, 0.15, 0.30):
    if part:
        touchees = ytr.sample(frac=part, random_state=0).index
        y_appris = ytr.copy()
        y_appris.loc[touchees] = 1 - y_appris.loc[touchees]
    else:
        y_appris = ytr

    ligne = {"étiquettes brouillées": f"{int(100 * part)} %"}

    # l'ensemble d'arbres, celui de la séance précédente : il apprend peu — le bruit le touche peu
    m = HistGradientBoostingClassifier(random_state=0).fit(Xtr, y_appris)
    ligne["ensemble d'arbres (%)"] = round(100 * m.score(Xval, yval), 2)

    # un arbre seul, sans limite de profondeur : il apprend tout, y compris ce qu'on vient de
    # brouiller. C'est celui de la démonstration « un arbre seul apprend la table par cœur ».
    t = DecisionTreeClassifier(random_state=0).fit(Xtr_n, y_appris)
    ligne["arbre seul, sans limite (%)"] = round(100 * t.score(Xval_n, yval), 2)

    brouille.append(ligne)

print("la même table, des étiquettes abîmées à des degrés différents \n")
display(pd.DataFrame(brouille))
print("→ le bruit d'étiquette ne mord pas sur un modèle qui apprend peu : il mord sur celui qui")
print("  apprend tout. C'est un signe qui ne trompe pas — et il se lit sur deux colonnes.")


> **Question →** `Devoir_maison_2.ipynb`, **question 5 (carnet J3) (2/2)** — le bruit d'étiquette, et le modèle qu'il abîme.


### Un trou n’est pas toujours un trou

Un trou dans une colonne n’a pas toujours la même valeur. Parfois la valeur manque **au hasard**.
Le trou ne dit rien de la ligne, il dit seulement qu’on ne l’a pas notée. Parfois, au contraire,
le fait qu’elle manque est en soi une information. Le combler, c’est effacer cette information.

La règle qui suit : **avant d’imputer, on regarde ce que le trou vaut.** C’est la décision la plus
fréquente du métier. Les débutants la prennent à l’envers : ils comblent d’abord, ils regardent
ensuite.

Ce que cette section **ne tranche pas** : quelle décision prendre pour une colonne donnée. Elle se
tranche colonne par colonne.


#### La méthode : ce que vaut un trou, avant de le combler

On ne devine pas si un trou est aléatoire : on le **mesure**. Pour une colonne, on compare le
taux d’absence des lignes où elle **manque** à celui des lignes où elle est **remplie**. S’ils se
ressemblent, la valeur manque sans rien dire : c’est **MCAR** (*missing completely at random*).
S’ils s’écartent, le fait de manquer porte de l’information : c’est **MNAR** (*missing not at
random*). Le vocabulaire est celui du métier, et c’est lui qui sert à écrire la décision.

Deux règles avant de comparer. Un : **on ne compare que des groupes d’au moins 500 lignes**, parce
qu’un groupe de trente lignes a un taux qui bouge tout seul. Deux : on nomme **le sens** du taux
qu’on cite.

« 75 % de trous chez les absents » et « 32 % d’absents chez les lignes à trou » ne sont pas la même
phrase. La première lit le trou *sachant la réponse*. La seconde lit la réponse *sachant le trou*.
C’est la seconde qui sert à décider : en production, le trou est connu et la réponse ne l’est pas.

Attention aussi : **structuré ne veut pas dire fuite**. Un trou structuré dont l’origine n’est pas
la réponse est une information légitime, qu’on garde.


In [ ]:
# la méthode, sur les deux colonnes que cette section traite
def etat_du_trou(table, colonne, plancher=500):
    """Les deux taux d'absence et les deux effectifs — refusés sous le plancher du cours."""
    manque = table[colonne].isna()
    n_manque, n_rempli = int(manque.sum()), int((~manque).sum())
    if min(n_manque, n_rempli) < plancher:
        return None
    return {'colonne': colonne,
            'n si ça manque': n_manque, 'n si c’est rempli': n_rempli,
            'taux d’absence si manque (%)': round(100 * table.loc[manque, CIBLE].mean(), 2),
            'taux d’absence si rempli (%)': round(100 * table.loc[~manque, CIBLE].mean(), 2)}

a_trous = [c for c in train.columns if train[c].isna().any()]
trop_fins = [c for c in a_trous if min(int(train[c].isna().sum()), int(train[c].notna().sum())) < 500]
print(f"le taux d'absence global, pour situer : {100 * train[CIBLE].mean():.2f} % \n")
print("les colonnes écartées du tableau (un groupe sous le plancher de 500) :", trop_fins)
display(pd.DataFrame([etat_du_trou(train, c) for c in ("solde_credit", "constantes_relevees")]))
print("→ deux taux qui se ressemblent : trou aléatoire (MCAR) ; deux taux qui s'écartent : trou")
print("  structuré (MNAR) — et ce trou-là porte de l'information, qu'imputer efface.")


In [ ]:
# un trou aléatoire : cinq traitements, et ce que chacun change
def score_de(Xe, ye=None):
    """Entraîne et mesure, toujours sur la même découpe."""
    ye = y if ye is None else ye
    Xtr_m, Xval_m, ytr_m, yval_m = train_test_split(Xe, ye, test_size=0.2, stratify=ye, random_state=0)
    return 100 * HistGradientBoostingClassifier(random_state=0).fit(Xtr_m, ytr_m).score(Xval_m, yval_m)

# le même découpage, les lignes retirées seulement côté apprentissage : le jeu de contrôle
# ne change pas, donc les deux scores se comparent vraiment — ce que `score_de` ne fait pas
def score_sans_lignes(masque):
    """Retire des lignes du seul côté apprentissage, et mesure sur le même jeu de contrôle."""
    garde = masque.loc[Xtr.index]
    return 100 * HistGradientBoostingClassifier(random_state=0).fit(
        Xtr[garde], ytr[garde]).score(Xval, yval)

colonne = "solde_credit"
mediane = X.copy()
mediane[colonne] = mediane[colonne].fillna(mediane[colonne].median())
avec_indicateur = mediane.copy()
avec_indicateur[colonne + "_etait_vide"] = X[colonne].isna().astype(int)
masque = X[colonne].notna()

print(f"la colonne {colonne}, cinq traitements du même trou \n")
print(f"  laissé tel quel            {score_de(X):6.2f} %")
print(f"  colonne retirée            {score_de(X.drop(columns=[colonne])):6.2f} %")
print(f"  trou comblé par la médiane {score_de(mediane):6.2f} %")
print(f"  médiane + indicateur       {score_de(avec_indicateur):6.2f} %")
print(f"  lignes à trou supprimées   {score_sans_lignes(masque):6.2f} %")
print("\n→ un trou aléatoire ne vaut rien : les cinq chiffres se tiennent à quelques centièmes.")


In [ ]:
# un trou structurel : le même geste, un autre résultat
constantes = train["constantes_relevees"].fillna("").astype("category")
# le mode est calculé, jamais écrit à la main : c’est le choix qu’on dit avoir fait
mode_complet = constantes.mode()[0]

X_garde = X.copy()
X_garde["constantes_relevees"] = constantes

X_comble = X.copy()
# combler par le mode fusionne le trou avec la valeur qui existe déjà le plus souvent :
# « non renseigné » et « complet » deviennent indistinguables — c’est le geste qui efface
X_comble["constantes_relevees"] = constantes.astype(str).replace("", mode_complet).astype("category")

print("la même colonne, trois traitements du même trou \n")
print(f"  le trou gardé comme catégorie  {score_de(X_garde):6.2f} %")
print(f"  le trou comblé par le mode ({mode_complet})  {score_de(X_comble):6.2f} %")
print(f"  la colonne retirée             {score_de(X):6.2f} %")

print("\n→ ici, garder le trou vaut plus que le combler. Le trou porte de l’information :")
print("  imputer sans regarder l’avait effacée — et c’est ce que cette section mesure.")


#### Ce que ce trou est vraiment : une fuite

D’où vient ce trou ? Le générateur de cette table répond : « manquant structurel causé par la
cible ». **75 %** de trous chez les absents, **35 %** chez les présents. Le trou suit donc la
**réponse**. Et une colonne dont le trou suit la réponse est une **fuite** — la même famille que
`motif_absence`, en plus discret.

Comme le trou est produit dans les **deux** tables, elle ne se contente pas de gagner en interne :
elle gagne **aussi** au classement. C’est la seule fuite du cours qui **survit** au découpage.
Le contraste avec `canal_prise`, qui gagne en interne et perd au classement, est la leçon. Ce qui
la trahit n’est pas sa stabilité : c’est l’**origine** de son trou.

**La décision, donc :** cette colonne est **écartée** — comme la table de travail le fait dès la
mise en place. Le **+0,72 point** qu’elle rapporte quand on la garde n’est pas un gain. C’est la
mesure de ce qu’une fuite rapporte, et **ce chiffre n’est pas fiable**.

Ce que cette section **ne tranche pas** : si d’autres colonnes cachent la réponse de la même
façon. C’est la traque que le devoir demande, et elle se fait signe par signe.


> **Question →** `Devoir_maison_2.ipynb`, **question 6 (carnet J3)** — un trou aléatoire, un trou structuré.


### Les décisions qui ne se voient pas dans le score

Reste une famille de décisions, celles qu’on prend tous les jours sans se poser de question.
Supprimer une colonne constante. Écarter une colonne de bruit. Traiter un code sans documentation.
Retirer une ligne dupliquée. Retirer une valeur impossible.

On les mesure **toutes**, et le résultat est presque toujours le même : **le score ne bouge pas.**
Ce n’est pas une déception, c’est un résultat. **Le nettoyage change rarement le score. Il change
ce que vous pouvez défendre**, et ce qui survit quand les données changent.

Ce que cette section **ne tranche pas** : lesquelles de ces décisions sont obligatoires. Aucune ne
se joue sur le score.


In [ ]:
# les colonnes inutiles — la raison de chacune est dans la liste de la section
inutiles = [c for c in ("zone_clinique", "indice_meteo_jour", "score_territorial",
                        "age_mois", "cout_acte") if c in X.columns]

print("les colonnes qui n'apportent rien \n")
print("  toutes les colonnes      ", f"{score_de(X):6.2f} %")
print(f"  sans {len(inutiles)} colonnes inutiles", f"{score_de(X.drop(columns=inutiles)):6.2f} %")
print("\n→ les cinq colonnes écartées, et la raison de chacune :")
for c in inutiles:
    print("   ", c)


In [ ]:
# les catégories non documentées : un code arrive, personne ne sait ce qu'il vaut
if "code_assurance" in X.columns:
    inconnus = int(train["code_assurance"].isin([7, 9]).sum())
    print("les codes absents du dictionnaire publié \n")
    print("  lignes concernées :", inconnus, f"({100 * inconnus / len(train):.2f} %)")
    print("  colonne gardée    :", f"{score_de(X):6.2f} %")
    sans_code = score_de(X.drop(columns=["code_assurance"]))
    print("  colonne retirée   :", f"{sans_code:6.2f} %")
    print("\n→ la décision ne se joue pas sur le score, mais sur ce qu'on peut dire de ce code :")
    print("  le garder en sachant qu'il est inexpliqué, le rattacher, ou l'écarter — et pourquoi.")


In [ ]:
# les valeurs impossibles et les doublons
impossibles = train["age"] == -1
sans_doublons = train.drop_duplicates()

# les trois mesures sur le même découpage : `score_sans_lignes` ne retire que côté apprentissage
base_nettoyage = score_de(X)
sans_age = score_sans_lignes(~impossibles)
sans_double = score_sans_lignes(~train.duplicated())

print("les valeurs impossibles, et les doublons \n")
print(f"  rien touché                  {base_nettoyage:6.2f} %")
print(f"  âges impossibles retirés     {sans_age:6.2f} %")
print(f"  lignes dupliquées retirées   {sans_double:6.2f} %")

print(f"\n→ la correction des âges retire {int(impossibles.sum())} lignes ; l’écart tient dans")
print(f"  {abs(base_nettoyage - sans_age):.2f} point, soit {abs(base_nettoyage - sans_age) / barre:.2f} barre — sous la barre, donc : le score ne bouge pas.")
print("  On jette de l’information avec le défaut — corriger n’est pas gratuit.")
print(f"  Les doublons, eux, ne retirent rien ({int(train.duplicated().sum())} ligne) : le score")
print("  est identique, et c’est le résultat attendu — le défaut n’est pas dans cette table")
print("  (J1, 2.4). Une décision qui ne change rien se note quand même : « vérifié, rien à faire »")
print("  est une conclusion.")


> **Question →** `Devoir_maison_2.ipynb`, **question 7 (carnet J3) (1/2)** — les décisions qui ne se voient pas dans le score.


### Le journal de nettoyage, et le pipeline

Chaque décision est maintenant mesurée. On les rassemble dans **un tableau**, dans l’esprit du
tableau d’essais de la séance précédente : **décision · choix · effet mesuré**. C’est ce tableau
qu’on rend. C’est lui qui répond à la question d’un audit : *pourquoi avez-vous fait ça ?*

Puis on assemble : une seule fonction, de la table brute à la prédiction, qui porte toutes les
décisions. C’est le **pipeline** — et c’est ce que j’exécuterai.

Ce que cette section **ne tranche pas** : si ce pipeline est le bon découpage dans le temps. Il ne
l’est pas, et la dernière section le mesure.


In [ ]:
# le journal de nettoyage — ce qui est lu n’est pas le score, c’est le tableau
JOURNAL = []

def journal(decision, choix, raison, effet):
    """Une décision, le choix fait, la raison qu’on donnerait à un audit, et l’effet mesuré
       en points — avec la comparaison à la barre, qui dit si l’effet compte."""
    JOURNAL.append({"décision": decision, "choix": choix, "raison (sans le score)": raison,
                    "effet mesuré": round(effet, 2),
                    "> barre ?": "oui" if abs(effet) > barre else "non"})
    display(pd.DataFrame(JOURNAL))

base = score_de(X)
journal("un trou aléatoire", "gardé tel quel",
        "le taux d’absence ne dépend pas du fait que la valeur manque (MCAR)",
        score_de(X) - base)
journal("un trou structurel", "fuite — écartée",
        "le trou suit la réponse : la garder serait prédire avec la réponse",
        score_de(X_garde) - base)
journal("des colonnes inutiles", "écartées",
        "constante, bruit pur, ou redondante avec une autre colonne",
        score_de(X.drop(columns=inutiles)) - base)
journal("un code non documenté", "gardé, en le sachant inexpliqué",
        "aucun dictionnaire ne dit ce que valent 7 et 9 ; le retirer serait décider sans savoir",
        sans_code - base)
journal("une valeur impossible", "lignes retirées",
        "un âge de −1 n’est pas un âge : la valeur est fausse, pas manquante",
        sans_age - base)
journal("des lignes dupliquées", "vérifié, rien à faire",
        "aucune ligne dupliquée dans cette table (mesuré)",
        sans_double - base)


> **Question →** `Devoir_maison_2.ipynb`, **question 7 (carnet J3) (2/2)** — le journal de nettoyage, tel que vous le rendez.


In [ ]:
# le pipeline : de la table brute à la prédiction, en un seul endroit
# Même contrat que `predict` : une table brute entre, un 0/1 par ligne sort.
def nettoyer(brut):
    """Les décisions apprises sur la table d’apprentissage, appliquées à une table brute :
       colonnes inutiles écartées, colonnes trop fines écartées, et les catégories de
       l’apprentissage — jamais recalculées sur la table qu’on reçoit."""
    d = matrice(brut)[0]
    d = d.drop(columns=[c for c in inutiles if c in d.columns])
    d = d[[c for c in X.columns if c in d.columns]]
    for c in CAT:
        if c not in d.columns:
            continue
        d[c] = d[c].astype("category").cat.set_categories(X[c].cat.categories)
    return d

def pipeline(brut):
    """Reçoit une table brute et rend un tableau de 0/1 — la chaîne complète."""
    return modele_final.predict(nettoyer(brut)).astype(int)

X_propre = nettoyer(train)
modele_final = HistGradientBoostingClassifier(random_state=0).fit(X_propre, y)
print("prédictions sur 500 lignes brutes :", pipeline(test.head(500)).shape)
print(f"la table préparée : {X_propre.shape[1]} colonnes (les {len(inutiles)} inutiles sont parties)")

soumission = pd.DataFrame({ID: test[ID], CIBLE: pipeline(test)})
soumission.to_csv("submission.csv", index=False)
print("le fichier écrit : submission.csv \n")
print("dimensions :", soumission.shape, "(lignes, colonnes)")


> **Question →** `Devoir_maison_2.ipynb`, **question 8 (carnet J3)** — le pipeline, et sa solidité.


### Et une chose qu’on ne traite pas encore : le temps

Nos deux tableaux ne sont pas tirés au hasard dans une population. Le fichier est un
**historique** : le test vient après l’entraînement dans le temps. Personne ne nous l’a dit. Il
faut le voir dans les dates, et le mesurer : c’est ce que fait la cellule ci-dessous, y compris là
où le découpage n’est pas parfait.

Pourquoi ça compte : un découpage **au hasard** sur un historique laisse le futur entrer dans
l’apprentissage. Le score monte, et il ne voudra rien dire. On y revient à la séance 5 —
« validation croisée et validation temporelle » : c’est là qu’on apprendra à découper sans se
mentir.

Ce que cette section **ne tranche pas** : comment découper sans se mentir. Le découpage temporel
s’apprend, il ne se devine pas.


In [ ]:
# la preuve, sans modèle : où s'arrête l'historique, où commence le test ?
historique = pd.to_datetime(train["date_rdv"])
a_venir = pd.to_datetime(test["date_rdv"])

print("la période de l'historique \n")
print("  train : du", historique.min().date(), "au", historique.max().date())
print("  test  : du", a_venir.min().date(), "au", a_venir.max().date())

# le découpage est chronologique — mais jusqu'où, exactement ?
borne = historique.max()
dedans = a_venir <= borne
apres = a_venir[~dedans]
print(f"\n→ {100 * (~dedans).mean():.2f} % des lignes de test viennent après l'historique :",
      f"du {apres.min().date()} au {apres.max().date()}")
print(f"  {int(dedans.sum())} ligne(s) de test portent une date antérieure à la borne.")
print()
print("Ces quelques lignes ne sont pas un détail à balayer : un découpage chronologique se vérifie,")
print("il ne se suppose pas. Elles ne changent rien à la leçon — un découpage au hasard, lui,")
print("mélangerait les deux périodes entièrement, et le score ne voudrait plus rien dire — mais")
print("elles disent exactement où le découpage est imparfait, et c'est ce qu'on attend d'un contrôle.")


### Ce qu’on retient

- **Un score se lit contre un concurrent nommé** : la réponse constante d’abord, la règle métier
  en place ensuite. Un écart se juge contre la **barre de bruit**, pas à l’œil.
- **Le public est un petit échantillon** : sur 1 000 lignes, l’erreur-type d’une exactitude vaut
  plus d’un point. Le classement qu’il donne ne se retrouve pas sur des lignes que personne n’a
  regardées.
- **Le tableau à quatre colonnes** (modèle · validation · public · écart) est ce qui rend l’écart
  lisible. Une colonne vide dit qu’un modèle n’a pas été soumis : ce n’est pas un oubli.
- **Un plafond existe**, et il n’est pas dans le modèle : trois familles très différentes donnent
  ici presque le même chiffre. On le mesure avant de chercher à l’atteindre.
- Un trou n’est pas toujours un trou. Aléatoire, il ne vaut rien. Structuré, il porte de
  l’information — donc on mesure avant d’imputer.
- Le nettoyage change rarement le score. Il change ce qu’on peut défendre, et ce qui survit quand
  les données changent. **Le journal de nettoyage** est ce qu’on rend.
- **Les deux tableaux ne sont pas tirés au hasard dans une population** : le fichier est un
  historique, et le test vient après. Un découpage au hasard y laisserait entrer le futur.

**Ce qu’on laisse ouvert :** le test **apparié** entre deux modèles jugés sur le même découpage.
La **calibration** des probabilités annoncées. Et le **découpage temporel**, qu’on ne sait pas
encore écrire sans se mentir.
